In [1]:
from pytao import TaoModel
from pprint import pprint
import json


In [2]:
M = TaoModel('/Users/chrisonian/Code/GitHub/lcls-lattice/bmad/models/cu_hxr/tao.init')

Initialized Tao with /var/folders/wj/lfgr01993dx79p9cm_skykbw0000gn/T/tmpdy7lmslt/tao/tao.init


In [3]:
%%tao
python lat_list 1@0>>Q*|base real:ele.s

-------------------------
Tao> python lat_list 1@0>>Q*|base real:ele.s
-------------------------
Tao> 


In [4]:
SLIST = M.cmd_real('python lat_list -track_only 1@0>>*|model real:ele.s')
LLIST = M.cmd_real('python lat_list -track_only  1@0>>*|model real:ele.l')
NAMES = M.cmd('python lat_list -track_only 1@0>>*|model ele.name')
IXLIST = [i for i in range(len(NAMES))]

ix_of = {}
for ix, name in enumerate(NAMES):
    if name in ix_of:
        pass
#        print("err", name)
    else: 
        ix_of[name] = ix

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

BEGINNING 0.0 0.0
DL00 -0.8690480000000003 -0.8690480000000003
LOADLOCK 0.8690480000000003 0.0
BEGGUN 0.0 0.0
SOL1BK 0.0 0.0
DBMARK80 0.0 0.0
CATHODE 0.0 0.0
DL01A 0.09600999999999998 0.09600999999999998
SOL1#1 0.1 0.19601
YC00 0.0 0.19601
XC00 0.0 0.19601
SQ01 0.0 0.19601
CQ01 0.0 0.19601
SOL1#2 0.1 0.29601
DL01A1 0.07851 0.37451999999999996
VV01 0.0 0.37451999999999996
DL01A2 0.11609 0.49061
AM00 0.0 0.49061
DL01A3 0.10461 0.59522
AM01 0.0 0.59522


In [5]:
# Search for split eles

def find_split_eles(slist, llist, names):
    """
    Searches for split elements
    
    """

    split_eles = {}
    basename = 'xxx'
    split_eles[basename] = {'stubs':[], 'splits':[], 'offsets':[]}
    stubs  = split_eles[basename]['stubs']
    splits = split_eles[basename]['splits']
    offsets = split_eles[basename]['offsets']
    s0 = 0
    for s, l, n in zip(SLIST, LLIST, NAMES):
        
        if l == 0:
            splits.append(n)
            offsets.append(s - s0)
        # Thick ele
        elif n.startswith(basename) and len(n)==len(basename)+1:
            # this is a true split ele     
            stubs.append(n)
            split_eles[basename]['L'] += l
        else:
            # New thick ele
            
            # Removing unwanted eles
            if basename == 'K21_3B':
                # Special case. Leave these
                print(basename, stubs, splits)
                pass
            elif len(stubs) == 1 or len(splits) == 0:
                split_eles.pop(basename)
            # Or if this is obviously a superimpose element
            elif basename.endswith('#'):
                split_eles.pop(basename)
                
            # Try a basename which excludes the final character
            basename = n[:-1]
            split_eles[basename] = {'stubs':[], 'splits':[], 'offsets':[]}
            split_eles[basename]['L'] = l
            s0 = s -l 
            stubs  = split_eles[basename]['stubs']
            splits = split_eles[basename]['splits']
            offsets = split_eles[basename]['offsets']
            stubs.append(n) # add this 
    
        #print (n, l, s, ix)
        
    split_eles.pop('xxx')        
    
    return split_eles

SPLIT_ELES = find_split_eles(SLIST, LLIST, NAMES)
    
pprint(SPLIT_ELES)

K21_3B ['K21_3B1', 'K21_3B2'] []
{'D10C': {'L': 3.0441000000000003,
          'offsets': [2.816100000000006],
          'splits': ['C29546'],
          'stubs': ['D10CE', 'D10CF']},
 'D21': {'L': 0.629435,
         'offsets': [0.5152000000000498],
         'splits': ['BL21'],
         'stubs': ['D21W', 'D21X', 'D21Y']},
 'D25CM': {'L': 0.5,
           'offsets': [0.26269999999999527],
           'splits': ['IM36'],
           'stubs': ['D25CMB', 'D25CMC']},
 'D38': {'L': 1.149823427686,
         'offsets': [0.609117535346968, 1.1498234276859876],
         'splits': ['XCDL4', 'YCDL4'],
         'stubs': ['D38B', 'D38C']},
 'D40CMW': {'L': 0.3915855,
            'offsets': [0.25154350000002523, 0.39158550000001924],
            'splits': ['BLTOF', 'MUHWALL1'],
            'stubs': ['D40CMWA', 'D40CMWB']},
 'DAQ10': {'L': 0.9089,
           'offsets': [0.13009999999997035,
                       0.3330999999999449,
                       0.5390999999999622,
                       0.776099

# Desplit klystrons

In [11]:
def lines_from_splits(name, split_ele_dict):
    offsets = split_ele_dict['offsets']
    splits = split_ele_dict['splits']
    stubs = split_ele_dict['stubs']
    L = round(split_ele_dict['L'], 9)
    
    

    
    lines = ['!---------------------', f'! {name} LCAVITY']

    
    ###     lines.append(f'{name}: {stubs[0]}, L = {L}')
    # These don't yet have full elements
    if name in ['L0A', 'L0B', 'L1X']:
        lines.append(f'{name}: {stubs[0]}, L = {L}')

    lines.append(f'{name}_full: line = ({name})')
    
    if len(splits) == 0:
        lines.append('! does not contain splitting elements. Consider reforming')
    else:
        lines.append('! contains zero length elements:')
    
    for n, o in zip(splits, offsets):
        o = round(o, 9)
        # This is at the end of the ele. Skip.
        if o == L:
            continue
        
        lines.append(f'    {n}[superimpose] = T')
        lines.append(f'    {n}[ref] = {name}')
        lines.append(f'    {n}[ref_origin] = beginning')
        lines.append(f'    {n}[offset] = {o}')
    lines.append('\n')
    
    return lines



CU_LINAC_REPLACEMENTS = {}

for name, ele in SPLIT_ELES.items():
    if any([name.startswith(x) for x in ['K', 'L0A', 'L0B', 'L1X'] ]) :
        
        # Strip off ___ from some of these
        name = name.strip('_')
        
        CU_LINAC_REPLACEMENTS[name] = '\n'.join(lines_from_splits(name, ele))
    
    
    
    #line = '\n'.join(lines_from_splits(name, ele))
    
with open('cu_linac_replacements.json', 'w') as outfile:
    json.dump(CU_LINAC_REPLACEMENTS, outfile, ensure_ascii=True, indent='  ')

In [12]:
pprint(CU_LINAC_REPLACEMENTS)

{'K21_1B': '!---------------------\n'
           '! K21_1B LCAVITY\n'
           'K21_1B_full: line = (K21_1B)\n'
           '! contains zero length elements:\n'
           '    XC21101[superimpose] = T\n'
           '    XC21101[ref] = K21_1B\n'
           '    XC21101[ref_origin] = beginning\n'
           '    XC21101[offset] = 0.453432\n'
           '    YC21102[superimpose] = T\n'
           '    YC21102[ref] = K21_1B\n'
           '    YC21102[ref_origin] = beginning\n'
           '    YC21102[offset] = 0.453432\n'
           '\n',
 'K21_1C': '!---------------------\n'
           '! K21_1C LCAVITY\n'
           'K21_1C_full: line = (K21_1C)\n'
           '! contains zero length elements:\n'
           '    XC21135[superimpose] = T\n'
           '    XC21135[ref] = K21_1C\n'
           '    XC21135[ref_origin] = beginning\n'
           '    XC21135[offset] = 0.4534458\n'
           '    YC21136[superimpose] = T\n'
           '    YC21136[ref] = K21_1C\n'
           '    YC21136[ref

In [13]:
!mv cu_linac_replacements.json good_cu_linac_replacements.json